# 06 — Final Training, Evaluation & Export (Stage 2)

**Purpose:** Train the best configuration identified from ablation studies
(notebooks 04-05), run comprehensive evaluation, and export the model
for deployment.

**Outputs:**
- Final checkpoint (`.pth`)
- ONNX export (`.onnx`)
- TorchScript export (`.pt`)
- Comprehensive evaluation report (JSON + plots)

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm opencv-python-headless tensorboard onnx onnxruntime

import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 98.2 MB/s eta 0:00:00


In [3]:
import os
import sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)


import os
import sys
import json
import logging
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader

from lib.config import cfg
from lib.models import get_net
from lib.core import get_loss, validate
from lib.dataset import BddDataset
from lib.utils.drive_dataset import (
    ensure_local_dataset_from_drive,
    find_raw_bdd_root,
    resolve_bdd_images_100k_dir,
    resolve_bdd_labels_100k_dir,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Stage-1 default target: YOLOP baseline.
CONFIG = 'YOLOP'
CHECKPOINT_NAME = 'best_joint.pth'
RUN_EVAL = True
EXPORT_ONNX = True
EXPORT_TORCHSCRIPT = True

yaml_map = {
    'YOLOPv2-paper-no-da': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_paper_no_da.yaml'),
    'YOLOPv2-best-row': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_best_row.yaml'),
    'YOLOPv2-focal-only': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_focal_only_ablation.yaml'),
    'YOLOP': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolop_vehicle_lane_baseline.yaml'),
}
run_name_map = {
    'YOLOPv2-paper-no-da': 'yolopv2_paper_no_da',
    'YOLOPv2-best-row': 'yolopv2_best_row',
    'YOLOPv2-focal-only': 'yolopv2_focal_only',
    'YOLOP': 'yolop',
}

cfg.defrost()
cfg.merge_from_file(yaml_map[CONFIG])

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)
BDD_IMAGES = resolve_bdd_images_100k_dir(RAW_BDD_ROOT)
BDD_LABELS = resolve_bdd_labels_100k_dir(RAW_BDD_ROOT)

cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = BDD_IMAGES
cfg.DATASET.LABELROOT = BDD_LABELS
cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')

run_name = run_name_map[CONFIG]
cfg.DRIVE.CHECKPOINT_DIR = os.path.join(REPO_ROOT, 'stage1', 'checkpoints', run_name)
cfg.DRIVE.METRICS_DIR = os.path.join(REPO_ROOT, 'stage1', 'metrics', run_name)
cfg.freeze()

checkpoint_candidates = [
    os.path.join(cfg.DRIVE.CHECKPOINT_DIR, CHECKPOINT_NAME),
    os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best_joint.pth'),
    os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth'),
    os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'latest.pth'),
]
ckpt_path = None
for cand in checkpoint_candidates:
    if os.path.exists(cand):
        ckpt_path = cand
        break
assert ckpt_path is not None, f'No checkpoint found in {cfg.DRIVE.CHECKPOINT_DIR}'

model = get_net(cfg).to(device)
model.gr = 1.0
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

print(f'[{CONFIG}] checkpoint: {ckpt_path}')
print(f'  epoch={ckpt.get("epoch", "?")}  nc={model.nc}  names={model.names}')
print(f'  dataset_root={cfg.DATASET.ROOT}')
print(f'  val_image_size={tuple(cfg.TEST.IMAGE_SIZE)}')


ModuleNotFoundError: No module named 'lib'

In [5]:

# ── Full validation first (stage-1 protocol) ──
eval_summary = None
if RUN_EVAL:
    val_wh = tuple(getattr(cfg.TEST, 'IMAGE_SIZE', [640, 384]))
    val_size = (int(val_wh[1]), int(val_wh[0]))

    val_dataset = BddDataset(cfg, is_train=False, inputsize=val_size, transform=T.ToTensor())
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.TEST.BATCH_SIZE_PER_GPU,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        collate_fn=val_dataset.collate_fn,
    )

    logger = logging.getLogger(f'eval_{CONFIG}')
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        logger.addHandler(logging.StreamHandler())

    criterion = get_loss(cfg, device)
    output_dir = os.path.join(cfg.DRIVE.METRICS_DIR, 'final_eval')
    os.makedirs(output_dir, exist_ok=True)

    ll_seg_result, det_result, val_loss, maps, times = validate(
        epoch=ckpt.get('epoch', -1),
        config=cfg,
        val_loader=val_loader,
        val_dataset=val_dataset,
        model=model,
        criterion=criterion,
        output_dir=output_dir,
        tb_log_dir='',
        writer_dict=None,
        logger=logger,
        device=device,
    )

    ll_acc, ll_iou, ll_miou = ll_seg_result
    mp, mr, map50, map_all = det_result
    eval_summary = {
        'config': CONFIG,
        'checkpoint': ckpt_path,
        'epoch': int(ckpt.get('epoch', -1)),
        'detection': {
            'precision': float(mp),
            'recall': float(mr),
            'mAP50': float(map50),
            'mAP50_95': float(map_all),
        },
        'lane': {
            'accuracy': float(ll_acc),
            'IoU': float(ll_iou),
            'mIoU': float(ll_miou),
        },
        'timing': {
            'inference_ms_per_image': float(times[0] * 1000.0),
            'nms_ms_per_image': float(times[1] * 1000.0),
        },
        'val_loss': float(val_loss),
    }

    summary_path = os.path.join(output_dir, f'{CONFIG.lower()}_{os.path.basename(ckpt_path).replace(".pth", "")}_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(eval_summary, f, indent=2)

    print('\n' + '=' * 70)
    print('STAGE-1 FINAL VALIDATION SUMMARY')
    print('=' * 70)
    print(json.dumps(eval_summary, indent=2))
    print(f'Saved summary to: {summary_path}')


onnxscript not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 21.4 MB/s eta 0:00:00


/tmp/ipykernel_1272/2313804035.py:21: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0419 02:46:48.176000 1272 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0419 02:46:49.605000 1272 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, a

[torch.onnx] Obtain model graph for `YOLOPv2DemoWrapper([...]` with `torch.export.export(..., strict=False)`...


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/usr/lib/python3.12/contextlib.py:144: UserWarning: The tensor attributes self.m.model.24.grid[0], self.m.model.24.grid[1], self.m.model.24.grid[2] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `YOLOPv2DemoWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


...

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


W0419 02:47:07.218000 1272 torch/onnx/_internal/exporter/_core.py:1248] Tensor 'm.model.24.stride' is not one of the initializers


[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:64: adapter_lookup: Assertion `false`

Applied 155 of general pattern rewrite rules.
ONNX exported to /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolop/exports/yolop.onnx
ONNX model validation passed.


In [6]:

import os

try:
    import onnxscript
except ImportError:
    print('onnxscript not found. Installing...')
    !pip install -q onnxscript
    import onnxscript

class DemoExportWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        det_out, lane_prob = self.m.predict(x)
        if isinstance(det_out, (tuple, list)):
            det_export = det_out[0] if isinstance(det_out, tuple) else det_out[0]
        else:
            det_export = det_out
        return det_export, lane_prob

export_model = DemoExportWrapper(model).to(device).eval()
export_dir = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'exports')
os.makedirs(export_dir, exist_ok=True)

dummy_input = torch.randn(1, 3, 640, 640, device=device)

if EXPORT_ONNX:
    onnx_path = os.path.join(export_dir, f'{CONFIG.lower()}_{os.path.basename(ckpt_path).replace(".pth", "")}.onnx')
    torch.onnx.export(
        export_model,
        dummy_input,
        onnx_path,
        opset_version=18,
        input_names=['images'],
        output_names=['det_out', 'lane_prob'],
        dynamic_axes={
            'images': {0: 'batch'},
            'det_out': {0: 'batch'},
            'lane_prob': {0: 'batch'},
        },
    )
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print(f'ONNX exported to {onnx_path}')

if EXPORT_TORCHSCRIPT:
    ts_path = os.path.join(export_dir, f'{CONFIG.lower()}_{os.path.basename(ckpt_path).replace(".pth", "")}.pt')
    traced = torch.jit.trace(export_model, dummy_input, strict=False, check_trace=False)
    traced.save(ts_path)
    print(f'TorchScript exported to {ts_path}')


/usr/local/lib/python3.12/dist-packages/torch/jit/_trace.py:1210: TracerWarning: Encountering a list at the output of the tracer might cause the trace to be incorrect, this is only valid if the container structure does not change based on the module's inputs. Consider using a constant container instead (e.g. for `list`, use a `tuple` instead. for `dict`, use a `NamedTuple` instead). If you absolutely need this and know the side effects, pass strict=False to trace() to allow this behavior.
  module._c._create_method_from_trace(
/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/lib/models/common.py:201: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.grid[i].shape[2:4] != x[i].shape[2:4]:


TracingCheckError: Tracing failed sanity checks!
ERROR: Graphs differed across invocations!
	Graph diff:
		  graph(%self.1 : __torch__.YOLOPv2DemoWrapper,
		        %x.1 : Tensor):
		    %m : __torch__.lib.models.yolop_baseline.MCnet = prim::GetAttr[name="m"](%self.1)
		    %model : __torch__.torch.nn.modules.container.Sequential = prim::GetAttr[name="model"](%m)
		    %_33 : __torch__.lib.models.common.Conv = prim::GetAttr[name="33"](%model)
		    %m.97 : __torch__.lib.models.yolop_baseline.MCnet = prim::GetAttr[name="m"](%self.1)
		    %model.65 : __torch__.torch.nn.modules.container.Sequential = prim::GetAttr[name="model"](%m.97)
		    %_32 : __torch__.torch.nn.modules.upsampling.Upsample = prim::GetAttr[name="32"](%model.65)
		    %m.95 : __torch__.lib.models.yolop_baseline.MCnet = prim::GetAttr[name="m"](%self.1)
		    %model.63 : __torch__.torch.nn.modules.container.Sequential = prim::GetAttr[name="model"](%m.95)
		    %_31 : __torch__.lib.models.common.BottleneckCSP = prim::GetA

In [ ]:

# ── Model size summary ──
param_count = sum(p.numel() for p in model.parameters())
ckpt_size_mb = os.path.getsize(ckpt_path) / 1024 / 1024
print(f'Parameters:     {param_count / 1e6:.2f} M')
print(f'Checkpoint:     {ckpt_size_mb:.1f} MB')
print(f'Checkpoint dir: {cfg.DRIVE.CHECKPOINT_DIR}')
print(f'Metrics dir:    {cfg.DRIVE.METRICS_DIR}')
if RUN_EVAL and eval_summary is not None:
    print(f"Eval mAP@0.5:   {eval_summary['detection']['mAP50']:.4f}")
    print(f"Eval lane IoU:  {eval_summary['lane']['IoU']:.4f}")
